In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 171 nodes, deleted 253 relationships, completed after 40 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 6 ms.


### Node Rules 

In [23]:
from type_checking.environment import Environment

env = Environment("../dtgraph/type_checking/env.json")
# env = Environment("../dtgraph/type_checking/env_common_movies.json")


generate_humans = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(:Movie)
GENERATE
(x = (p):Actor {
    name = p.name,
    old = p.born
})
''',
env=env,
type_strict = True)

# common_movies = Rule('''
# MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)
# WHERE id(x) < id(z)
# GENERATE
# ((x):Actor {
#     name = x.name
# })-[(x,z):ACTED_WITH {
#     CommonMovies = [y.title]
# }]->((z):Actor {
#     name = z.name
# })
# ''',
# env=env,
# type_strict = True
# )


# generate_created = Rule('''
# MATCH (p:Person)-[:DIRECTED]->(m:Movie)
# GENERATE
# (x = (p):)-[():CREATED {
#     role = 123
# }]->(y = (m):)
# ''',
# env=env,
# type_strict = True)

### Execute Rules

In [22]:
my_transform = Transformation([generate_humans])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 6 ms.
AST:
PropertyAccess
    ├── var: p
    └── prop: name
AST:
PropertyAccess
    ├── var: p
    └── prop: born


CompileError: 
Type checking failed for rule:
Source dictionary:
{'lhs': 'MATCH (p:Person)-[:ACTED_IN]->(:Movie)', 'constructors': [{'alias': 'x', 'ids': ['p'], 'labels': ['Actor'], 'properties': [{'key': 'name', 'value': 'p.name', 'ast': <type_checking.ast_nodes.PropertyAccess object at 0x00000140DB4DB1F0>}, {'key': 'old', 'value': 'p.born\n', 'ast': <type_checking.ast_nodes.PropertyAccess object at 0x00000140DB18BC10>}]}]}

Reason: Unknown source property: born

### Abort Transformation

In [ ]:
my_transform.abort()